# PerfectPetClub ETL
Extract the database, CRM, server log and survey data, build one row per customer, and save it as `customer_churn.parquet`.

## 1. Setup

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

def find_project_dir():
    """Return (project folder, data folder), whichever of Data/data exists."""
    here = Path.cwd().resolve()
    for base in (here, here.parent):
        for name in ("Data", "data"):
            if (base / name).is_dir():
                return base, base / name
    raise FileNotFoundError("Data folder not found. Open the notebook from the project or notebooks folder.")

PROJECT_DIR, DATA_DIR = find_project_dir()

DB_PATH = DATA_DIR / "PerfectPetClub.db"
CRM_PATH = DATA_DIR / "PerfectPetClub_CRM_report.csv"
SERVER_PATH = DATA_DIR / "PerfectPetClub_server_log.csv"
SURVEY_PATH = DATA_DIR / "PerfectPetClub_survey.csv"
OUTPUT_PATH = DATA_DIR / "customer_churn.parquet"

print("Data folder:", DATA_DIR)
for path in (DB_PATH, CRM_PATH, SERVER_PATH, SURVEY_PATH):
    print(path.name, "-", "found" if path.exists() else "MISSING")

Data folder: C:\Users\Admin\P3G7-PerfectPetClub-Customer-Churn\Data
PerfectPetClub.db - found
PerfectPetClub_CRM_report.csv - found
PerfectPetClub_server_log.csv - found
PerfectPetClub_survey.csv - found


## 2. Get the survey file from S3 (only if it is missing)
Needs the university VPN

In [8]:
%pip install boto3 pyarrow

Note: you may need to restart the kernel to use updated packages.


In [12]:
import boto3
from getpass import getpass
from botocore.exceptions import ClientError, EndpointConnectionError

ENDPOINT_URL = "http://vuh-la-specs-s3-1.herts.ac.uk:9000"
BUCKET_NAME = "comsci-student-s3-1"

ACCESS_KEY = getpass("Enter your S3 access key: ")
SECRET_KEY = getpass("Enter your S3 secret key: ")

s3 = boto3.client(
    "s3",
    endpoint_url=ENDPOINT_URL,
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name="us-east-1"
)

try:
    response = s3.list_objects_v2(
        Bucket=BUCKET_NAME,
        MaxKeys=100
    )

    print("Successfully connected to S3!")

except EndpointConnectionError:
    print("Connection failed. Check your UH VPN and endpoint.")

except ClientError as error:
    print("S3 error:", error)

Enter your S3 access key:  ········
Enter your S3 secret key:  ········


Successfully connected to S3!


In [35]:
# List files stored in the S3 bucket

paginator = s3.get_paginator("list_objects_v2")

print("Available files in S3 bucket:\n")

for page in paginator.paginate(Bucket=BUCKET_NAME):
    for item in page.get("Contents", []):
        print(
            "File:", item["Key"],
            "| Size:", round(item["Size"] / 1024, 2), "KB"
        )

Available files in S3 bucket:

File: PerfectPetClub_survey.csv | Size: 8.81 KB
File: TinyFile.txt | Size: 0.03 KB


In [36]:
from pathlib import Path

current_dir = Path.cwd()

if (current_dir / "data").exists():
    PROJECT_DIR = current_dir
elif (current_dir.parent / "data").exists():
    PROJECT_DIR = current_dir.parent
else:
    raise FileNotFoundError(
        "Cannot find the project data folder. Check your notebook location."
    )

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)

Project directory: C:\Users\Admin\P3G7-PerfectPetClub-Customer-Churn
Data directory: C:\Users\Admin\P3G7-PerfectPetClub-Customer-Churn\data


In [37]:
# Download survey dataset from S3

OBJECT_KEY = "PerfectPetClub_survey.csv"

SURVEY_PATH = DATA_DIR / "PerfectPetClub_survey.csv"

s3.download_file(
    BUCKET_NAME,
    OBJECT_KEY,
    str(SURVEY_PATH)
)

print("Survey dataset downloaded successfully!")
print("Saved location:", SURVEY_PATH)

Survey dataset downloaded successfully!
Saved location: C:\Users\Admin\P3G7-PerfectPetClub-Customer-Churn\data\PerfectPetClub_survey.csv


In [38]:
import pandas as pd

survey_data = pd.read_csv(SURVEY_PATH)

print("Survey dataset loaded successfully")

print("\nDataset dimensions:")
print(survey_data.shape)

print("\nColumn names:")
print(survey_data.columns.tolist())

print("\nFirst five records:")
display(survey_data.head())

Survey dataset loaded successfully

Dataset dimensions:
(221, 5)

Column names:
['email', 'reported_income', 'nps_recommend', 'satisfaction_score', 'online_comparison_freq']

First five records:


,email,reported_income,nps_recommend,satisfaction_score,online_comparison_freq
0,Sargsyan701@somemail.co.uk,26k-30k,8,6,5
1,Mammadova282@example.com,34k-38k,5,8,9
2,Sana858@notrealemail.com,26k-30k,9,5,5
3,Omar.Farooq4@example.com,198k-200k+,9,3,5
4,Alejandro357@somemail.co.uk,42k-46k,0,8,10


In [40]:
print("Dataset information:")
survey_data.info()

print("\nMissing values:")
display(survey_data.isnull().sum())

print("\nDuplicate records:")
print(survey_data.duplicated().sum())

print("\nStatistical summary:")
display(survey_data.describe(include="all").T)

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 221 entries, 0 to 220
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   email                   221 non-null    object
 1   reported_income         221 non-null    object
 2   nps_recommend           221 non-null    int64 
 3   satisfaction_score      221 non-null    int64 
 4   online_comparison_freq  221 non-null    int64 
dtypes: int64(3), object(2)
memory usage: 8.8+ KB

Missing values:


email                     0
reported_income           0
nps_recommend             0
satisfaction_score        0
online_comparison_freq    0
dtype: int64


Duplicate records:
0

Statistical summary:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
email,221,221,Sargsyan701@somemail.co.uk,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
reported_income,221,20,34k-38k,33,NaN,NaN,NaN,NaN,NaN,NaN,NaN
nps_recommend,221.0,NaN,NaN,NaN,5.80543,2.883049,0.0,3.0,6.0,8.0,10.0
satisfaction_score,221.0,NaN,NaN,NaN,4.760181,3.34276,0.0,2.0,5.0,8.0,10.0
online_comparison_freq,221.0,NaN,NaN,NaN,6.090498,2.826972,0.0,4.0,6.0,9.0,10.0


In [42]:
required_files = [
    "PerfectPetClub.db",
    "PerfectPetClub_CRM_report.csv",
    "PerfectPetClub_server_log.csv",
    "PerfectPetClub_survey.csv"
]

print("Project dataset verification:\n")

for filename in required_files:
    file_path = DATA_DIR / filename

    if file_path.exists():
        print("FOUND:", filename)
    else:
        print("MISSING:", filename)

Project dataset verification:

FOUND: PerfectPetClub.db
FOUND: PerfectPetClub_CRM_report.csv
FOUND: PerfectPetClub_server_log.csv
FOUND: PerfectPetClub_survey.csv


In [43]:
PROJECT_DIR = Path(PROJECT_DIR)

for file in PROJECT_DIR.rglob("PerfectPetClub.db"):
    print("Database found:", file)

Database found: C:\Users\Admin\P3G7-PerfectPetClub-Customer-Churn\data\PerfectPetClub.db


In [44]:
import sqlite3

DB_PATH = DATA_DIR / "PerfectPetClub.db"

if DB_PATH.exists():
    connection = sqlite3.connect(DB_PATH)
    print("Database connected successfully!")
    connection.close()
else:
    print("Database file is still missing.")

Database connected successfully!


In [23]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
import logging

print("Libraries imported successfully!")

Libraries imported successfully!


In [24]:
DB_PATH = DATA_DIR / "PerfectPetClub.db"

connection = sqlite3.connect(DB_PATH)

print("Connected to PerfectPetClub database!")

Connected to PerfectPetClub database!


In [25]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table';
    """,
    connection
)

print("Tables available in database:")
display(tables)

Tables available in database:


,name
0,payment
1,sqlite_sequence
2,product_category
3,product
4,subscription_type
5,subscription
6,customer
7,sale
8,sale_product
9,product_review


In [27]:
for table_name in tables["name"]:
    print("\n" + "=" * 60)
    print("TABLE:", table_name)
    print("=" * 60)

    table_data = pd.read_sql_query(
        f'SELECT * FROM "{table_name}" LIMIT 5',
        connection
    )

    display(table_data)

    print("Columns:")
    print(table_data.columns.tolist())


TABLE: payment


,id,amount,payment_gateway,transaction_reference
0,1,299,paypool,41f6e886f30a470e83c86a8b3fb0602f
1,2,299,cardpay,cbd0263851ac45d395b009c895f4b673
2,3,1199,directdebit,50b41d8af9584ff098c3d51caae933ce
3,4,1199,paypool,7c936ce05b5549589066c9258afc3e2a
4,5,299,cardpay,eebe7fadfefa4572b59cd04531575765


Columns:
['id', 'amount', 'payment_gateway', 'transaction_reference']

TABLE: sqlite_sequence


,name,seq
0,subscription_type,3
1,product,194
2,product_category,26
3,subscription,221
4,customer,221


Columns:
['name', 'seq']

TABLE: product_category


,id,name
0,1,Feeding Accessories
1,2,Small Animal Food
2,3,Cooling Products
3,4,Housing & Cages
4,5,Cat Litter


... (11 outputs left out)

,id,quantity,sale_id,product_id
0,1,1,1,3
1,2,1,2,3
2,3,1,3,1
3,4,1,4,1
4,5,1,5,3


Columns:
['id', 'quantity', 'sale_id', 'product_id']

TABLE: product_review


,id,star_rating,product_id,customer_id,sale_id
0,1,1,100,31,56
1,2,4,143,17,58
2,3,1,63,47,59
3,4,3,8,4,60
4,5,1,156,1,63


Columns:
['id', 'star_rating', 'product_id', 'customer_id', 'sale_id']


In [45]:
datasets = {
    "CRM": crm_data,
    "Server Logs": server_data,
    "Survey": survey_data
}

for name, df in datasets.items():

    print("\n", "=" * 50)
    print(name)
    print("=" * 50)

    print("Rows and columns:", df.shape)
    print("Column names:", df.columns.tolist())

    display(df.head())


CRM
Rows and columns: (7585, 4)
Column names: ['customer_id', 'recieved_datatime', 'platform', 'type']


,customer_id,recieved_datatime,platform,type
0,55,2026-01-01 00:42:47.000,['e-mail'],complaint
1,49,2026-01-01 01:25:34.000,['live-chat'],query
2,44,2026-01-01 03:33:55.001,['telephone'],query
3,33,2026-01-01 05:42:16.002,['e-mail'],complaint
4,33,2026-01-01 06:25:03.002,['e-mail'],complaint



Server Logs
Rows and columns: (67249, 5)
Column names: ['datetime', 'session_id', 'ip', 'login', 'user_agent']


,datetime,session_id,ip,login,user_agent
0,2026-01-01 02:08:21.000,session_cookie::63fec84dedaa4f2cbaa10b413b5a7d...,239.66.231.182,Sipho928@somemail.co.uk,"""Mozilla/5.0 (Desktop/Laptop; Cipher CoreBook ..."
1,2026-01-01 02:51:08.001,session_cookie::64bcd32ed1494bd1af6b7a233cd638...,200.200.140.49,Sienna.Martin923@notrealemail.com,"""Mozilla/5.0 (Mobile; Pulse PulsePhone) AppleW..."
2,2026-01-01 04:16:42.001,session_cookie::f848746c9c96422b9eb050eccacfeb...,38.219.218.103,v134@somemail.co.uk,"""Mozilla/5.0 (Mobile; Titan Pocket Go) AppleWe..."
3,2026-01-01 04:59:29.002,session_cookie::4589deeb679c4ae68f62a1f61af2db...,126.234.10.95,v134@somemail.co.uk,"""Mozilla/5.0 (Mobile; Titan Pocket Go) AppleWe..."
4,2026-01-01 07:07:50.003,session_cookie::2bc1377dd7a74949be4e0035137f5f...,17.99.243.42,io415@fakemail.co.uk,"""Mozilla/5.0 (Tablet; Lumina CorePad) AppleWeb..."



Survey
Rows and columns: (221, 5)
Column names: ['email', 'reported_income', 'nps_recommend', 'satisfaction_score', 'online_comparison_freq']


,email,reported_income,nps_recommend,satisfaction_score,online_comparison_freq
0,Sargsyan701@somemail.co.uk,26k-30k,8,6,5
1,Mammadova282@example.com,34k-38k,5,8,9
2,Sana858@notrealemail.com,26k-30k,9,5,5
3,Omar.Farooq4@example.com,198k-200k+,9,3,5
4,Alejandro357@somemail.co.uk,42k-46k,0,8,10


In [46]:
connection.close()

print("Database connection closed.")

Database connection closed.


## 3. Extract

In [28]:
def extract_sqlite_tables(db_path):
    """Read every table in the SQLite database into a dict of DataFrames."""
    with sqlite3.connect(db_path) as conn:
        names = pd.read_sql_query(
            "SELECT name FROM sqlite_master WHERE type = 'table';", conn
        )["name"].tolist()
        return {n: pd.read_sql_query(f'SELECT * FROM "{n}"', conn) for n in names}


def extract_csv(path):
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    return pd.read_csv(path)


database_data = extract_sqlite_tables(DB_PATH)
crm_data = extract_csv(CRM_PATH)
server_data = extract_csv(SERVER_PATH)
survey_data = extract_csv(SURVEY_PATH)

for name, table in database_data.items():
    print(f"db.{name}: {table.shape}")
for name, table in [("crm", crm_data), ("server log", server_data), ("survey", survey_data)]:
    print(f"{name}: {table.shape}")

db.payment: (23188, 4)
db.sqlite_sequence: (9, 2)
db.product_category: (26, 2)
db.product: (194, 4)
db.subscription_type: (3, 3)
db.subscription: (221, 5)
db.customer: (221, 9)
db.sale: (23188, 6)
db.sale_product: (59759, 4)
db.product_review: (10990, 5)
crm: (7585, 4)
server log: (67249, 5)
survey: (221, 5)


In [30]:
def extract_csv(file_path):
    """
    Extract a CSV file into a Pandas DataFrame.
    """
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    return pd.read_csv(file_path)


crm_data = extract_csv(DATA_DIR / "PerfectPetClub_CRM_report.csv")

server_data = extract_csv(DATA_DIR / "PerfectPetClub_server_log.csv")

survey_data = extract_csv(DATA_DIR / "PerfectPetClub_survey.csv")

print("All CSV datasets extracted successfully!")

All CSV datasets extracted successfully!


## 4. Transform
Churn is defined as auto-renewal switched off (`auto_renew == 0`). Each source is reduced to one row per customer before joining.

In [31]:
def build_customer_master(db):
    """Customers joined to their subscription and subscription type, with the churn flag."""
    customers = db["customer"].drop(columns=["password_hash"], errors="ignore")

    master = customers.merge(
        db["subscription"],
        left_on="subscription_id", right_on="id",
        how="left", suffixes=("_customer", "_subscription"), validate="one_to_one",
    )
    master = master.merge(
        db["subscription_type"],
        left_on="subscription_type_id", right_on="id",
        how="left", suffixes=("", "_type"), validate="many_to_one",
    )

    master["auto_renew"] = pd.to_numeric(master["auto_renew"], errors="coerce")
    if master["auto_renew"].isna().any():
        raise ValueError("Missing auto_renew values need investigation.")
    unexpected = set(master["auto_renew"].unique()) - {0, 1}
    if unexpected:
        raise ValueError(f"Unexpected auto_renew values: {unexpected}")

    master["churn"] = (master["auto_renew"] == 0).astype(int)
    master["customer_id"] = master["id_customer"]
    return master


def clean_email(series):
    return series.astype("string").str.strip().str.lower()


def build_sales_features(db):
    sales = db["sale"].merge(
        db["payment"][["id", "amount"]],
        left_on="payment_id", right_on="id",
        how="left", suffixes=("", "_payment"), validate="many_to_one",
    )
    items = (
        db["sale_product"].groupby("sale_id")["quantity"].sum()
        .reset_index(name="items_purchased")
    )
    sales = sales.merge(items, left_on="id", right_on="sale_id", how="left", validate="one_to_one")

    return sales.groupby("customer_id").agg(
        total_orders=("id", "nunique"),
        total_payment_amount=("amount", "sum"),
        average_payment_amount=("amount", "mean"),
        total_items_purchased=("items_purchased", "sum"),
    ).reset_index()


def build_crm_features(crm_raw):
    # the source column name has a typo
    crm = crm_raw.rename(columns={"recieved_datatime": "received_datetime"})
    crm["type"] = crm["type"].astype(str).str.strip().str.lower()

    return crm.groupby("customer_id").agg(
        total_crm_interactions=("type", "count"),
        complaint_count=("type", lambda x: (x == "complaint").sum()),
        query_count=("type", lambda x: (x == "query").sum()),
    ).reset_index()


def build_server_features(server_raw):
    server = server_raw.copy()
    server["email_key"] = clean_email(server["login"])
    return server.groupby("email_key").agg(
        total_logins=("session_id", "count"),
        unique_sessions=("session_id", "nunique"),
    ).reset_index()


def build_survey_features(survey_raw):
    survey = survey_raw.copy()
    survey["email_key"] = clean_email(survey["email"])
    if survey["email_key"].duplicated().any():
        raise ValueError("Duplicate survey emails found. Check before merging.")
    return survey.drop(columns=["email"])

In [32]:
customer_master = build_customer_master(database_data)
customer_master["email_key"] = clean_email(customer_master["email"])

print("Churn rate: %.2f%%" % (customer_master["churn"].mean() * 100))

final_data = (
    customer_master
    .merge(build_sales_features(database_data), on="customer_id", how="left", validate="one_to_one")
    .merge(build_crm_features(crm_data), on="customer_id", how="left", validate="one_to_one")
    .merge(build_server_features(server_data), on="email_key", how="left", validate="one_to_one")
    .merge(build_survey_features(survey_data), on="email_key", how="left", validate="one_to_one")
)
print("Merged shape:", final_data.shape)

Churn rate: 38.91%
Merged shape: (221, 32)


In [33]:
# Customers with no sales, CRM or login records have zero activity, not missing data
activity_columns = [
    "total_orders", "total_payment_amount", "average_payment_amount",
    "total_items_purchased", "total_crm_interactions", "complaint_count",
    "query_count", "total_logins", "unique_sessions",
]
final_data[activity_columns] = final_data[activity_columns].fillna(0)

# Drop personal details and join keys that are no longer needed
columns_to_remove = [
    "name", "email", "email_key", "address_line_1", "address_line_2",
    "county", "postcode", "id_customer", "id_subscription", "id_type", "auto_renew",
]
final_data = final_data.drop(columns=columns_to_remove, errors="ignore")

print("Missing values left:")
display(final_data.isnull().sum()[lambda s: s > 0])
print(final_data.columns.tolist())

Missing values left:


Series([], dtype: int64)

['subscription_id', 'start_date', 'end_date', 'subscription_type_id', 'id', 'name_type', 'delivery_service', 'churn', 'customer_id', 'total_orders', 'total_payment_amount', 'average_payment_amount', 'total_items_purchased', 'total_crm_interactions', 'complaint_count', 'query_count', 'total_logins', 'unique_sessions', 'reported_income', 'nps_recommend', 'satisfaction_score', 'online_comparison_freq']


## 5. Check and load

In [34]:
assert final_data["customer_id"].notna().all(), "Missing customer IDs"
assert final_data["customer_id"].is_unique, "Duplicate customer IDs"
assert final_data["churn"].isin([0, 1]).all(), "Invalid churn values"

def load_parquet(df, output_path):
    if df.empty:
        raise ValueError("Cannot save an empty dataset.")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(output_path, engine="pyarrow", index=False)
    print("Saved", output_path, df.shape)

load_parquet(final_data, OUTPUT_PATH)

# read it back as a final check
display(pd.read_parquet(OUTPUT_PATH).head())

Saved C:\Users\Admin\P3G7-PerfectPetClub-Customer-Churn\Data\customer_churn.parquet (221, 22)


,subscription_id,start_date,end_date,subscription_type_id,id,name_type,delivery_service,churn,customer_id,total_orders,...,total_items_purchased,total_crm_interactions,complaint_count,query_count,total_logins,unique_sessions,reported_income,nps_recommend,satisfaction_score,online_comparison_freq
0,1,2026-01-01 00:00:00.000,2026-01-31 00:00:00.000,3,3,SuperSaver,Economy72,1,1,20,...,46,3.0,2.0,1.0,48,48,26k-30k,8,6,5
1,2,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,3,3,SuperSaver,Economy72,0,2,253,...,748,45.0,21.0,24.0,1085,1085,34k-38k,5,8,9
2,3,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,1,Premium,SignedFor24,0,3,304,...,824,167.0,90.0,77.0,624,624,26k-30k,9,5,5
3,4,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,1,Premium,SignedFor24,0,4,1055,...,7362,281.0,132.0,149.0,1540,1540,198k-200k+,9,3,5
4,5,2026-01-01 00:00:00.000,2026-05-01 00:00:00.000,3,3,SuperSaver,Economy72,1,5,44,...,116,5.0,1.0,4.0,388,388,42k-46k,0,8,10
